# Lesson 17: Generative Recurrent Networks (Character & Word LSTM)

In this notebook, you will:
1. Train a **Character-Level LSTM** in PyTorch using Teacher Forcing.
2. Sample new text with **Temperature Scaling**.
3. Complete the **Lab**: Adapt the generator to **Word-Level** token sequences.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

text = "neural networks learn patterns from data and generate new sequences. " * 15
chars = sorted(set(text))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}

class CharLSTM(nn.Module):
    def __init__(self, vocab_size, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 32)
        self.lstm = nn.LSTM(32, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)
    def forward(self, x, state=None):
        out, state = self.lstm(self.emb(x), state)
        return self.fc(out), state

model = CharLSTM(len(chars))
opt = torch.optim.Adam(model.parameters(), lr=0.02)
seq = torch.tensor([c2i[c] for c in text], dtype=torch.long)
X, Y = seq[:-1].unsqueeze(0), seq[1:].unsqueeze(0)

for ep in range(40):
    opt.zero_grad()
    logits, _ = model(X)
    loss = F.cross_entropy(logits.view(-1, len(chars)), Y.view(-1))
    loss.backward()
    opt.step()
print(f"Final Training Loss: {loss.item():.4f}")

In [ ]:
def generate(model, seed="neural ", length=45, temp=0.7):
    model.eval()
    out = list(seed)
    inp = torch.tensor([[c2i[c] for c in seed]])
    with torch.no_grad():
        logits, state = model(inp)
        for _ in range(length):
            probs = F.softmax(logits[0, -1] / temp, dim=0)
            idx = torch.multinomial(probs, 1).item()
            out.append(i2c[idx])
            logits, state = model(torch.tensor([[idx]]), state)
    return "".join(out)

print("Generated (T=0.7):", generate(model, temp=0.7))